# Módulo 5 · Clase 3 — ¿Cuánto queda?

### Machine Learning for Petroleum Engineers Using Python
**SLB Ecuador · UDLA · 2026** — Carlos Enrique Mosquera Trujillo

Repositorio: https://github.com/cmosquerat/slb-diplomado

---

## La idea de hoy

Un campo pasó su pico hace **cinco años**. Gerencia tiene que decidir si sigue
invirtiendo o lo prepara para abandono.

**¿Cuánto petróleo le queda por producir en los próximos siete años?**

1. Miramos los datos con calma: 54 campos del Mar del Norte, hasta 49 años de historia.
2. Ajustamos **Arps exponencial** — la recta de la Clase 1, que ahora tiene nombre.
3. Le medimos el defecto en los 54 campos: no falla al azar, **falla siempre para el
   mismo lado**.
4. Probamos la **hiperbólica**, y vemos si se gana el lugar.
5. Y al final, lo que de verdad se entrega: una **banda auditada**.

> **Lo que hace especial a esta clase:** los 54 campos **ya produjeron** esos 12 años.
> Podemos tapar, pronosticar, destapar y ver quién tenía razón. Es una auditoría,
> no una simulación.

---

# 0 · Preparación

In [ ]:
# pandas: tablas.                                        (Modulo 1)
import pandas as pd

# numpy: cuentas con muchos numeros a la vez.            (Modulo 1)
import numpy as np

# matplotlib: graficos.                                  (Modulo 1)
import matplotlib.pyplot as plt

# curve_fit: busca los parametros de una formula que no
# es una recta. Es lo unico nuevo de hoy.
from scipy.optimize import curve_fit

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Listo.")

In [ ]:
# el archivo se baja solo desde GitHub
URL = ("https://raw.githubusercontent.com/cmosquerat/slb-diplomado/"
       "main/datos/campos_noruega_declinacion.csv")

d = pd.read_csv(URL)

print("filas y columnas:", d.shape)

---

# 1 · Exploración de los datos  ⏱️ *15–20 minutos*

Como siempre: **nada de modelos hasta entender qué tenemos**. Ocho preguntas, y cada
una deja una decisión escrita.

## 1.1 · ¿Cómo se ve una fila?

In [ ]:
# las primeras filas
d.head(5)

In [ ]:
# y las ultimas: siempre hay que mirar las dos puntas
d.tail(5)

## 1.2 · ¿Qué es cada columna?

| columna | unidad | qué es |
|---|---|---|
| `campo` | — | Nombre del campo. Es nuestra **unidad de análisis** |
| `fecha` | — | El mes de calendario |
| `mes_desde_pico` | meses | **El reloj que importa**: 0 es el mes de máxima producción |
| `dias` | días | Días de ese mes |
| `oil_bpd` | bbl/día | **Lo que vamos a modelar** |
| `agua_bpd` | bbl/día | Agua producida. Hoy no se usa — es la Clase 4 |

El dato original del regulador noruego viene en **millones de metros cúbicos estándar
por mes**. La conversión está en `preparar_datos.py`: un Sm³ son 6,2898 barriles, y se
divide por los días del mes.

In [ ]:
# de que tipo es cada columna
d.dtypes

## 1.3 · ¿Cuántos campos, y cuánta historia?

In [ ]:
print("campos:", d.campo.nunique())
print("filas: ", len(d))

# cuantos meses tiene cada campo despues de su pico
meses = d.groupby("campo").size()

# .describe() en anios, que se entiende mejor
print()
print((meses / 12).describe().round(0))

In [ ]:
# un histograma se lee mas rapido
plt.figure(figsize=(8, 3.2))
plt.hist(meses / 12, bins=14, color="#2563EB", alpha=0.75)
plt.xlabel("anios de historia despues del pico")
plt.ylabel("cantidad de campos")
plt.title("Cuanta historia tiene cada campo")
plt.show()

**Lectura**: la mediana son 25 años de historia después del pico. El más viejo tiene
49 años — es Ekofisk, que arrancó en 1971 y **sigue produciendo**.

> 🤔 **Pregunta clave**: si quisiéramos verificar un pronóstico a 12 años,
> ¿cuántos años de historia necesita como mínimo un campo para servirnos?

## 1.4 · El reloj que importa: desde el pico

Ekofisk arrancó en 1971 y Grane en 2003. En el eje de calendario no se parecen en nada.
Pero **la declinación no empieza en una fecha: empieza en el pico**.

In [ ]:
# comparamos los dos ejes: calendario contra "meses desde el pico"
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.6))

for nombre in ["EKOFISK", "GRANE", "DRAUGEN"]:
    g = d[d.campo == nombre]
    # a la izquierda: el eje de calendario
    ejes[0].plot(pd.to_datetime(g.fecha), g.oil_bpd / 1000, lw=1.3, label=nombre)
    # a la derecha: el reloj desde el pico
    ejes[1].plot(g.mes_desde_pico / 12, g.oil_bpd / 1000, lw=1.3, label=nombre)

ejes[0].set_title("Eje de calendario: no se parecen")
ejes[0].set_xlabel("anio")
ejes[1].set_title("Desde el pico: ahora si")
ejes[1].set_xlabel("anios desde el pico")
for e in ejes:
    e.set_ylabel("miles de bbl/dia")
    e.legend(fontsize=8)
plt.tight_layout()
plt.show()

### ¿Y cómo se encontró ese pico?

El archivo ya viene alineado, pero **la decisión de dónde poner el cero no es obvia** y
condiciona todo lo que sigue: qué cinco años vemos y con qué análogos nos comparamos.

El criterio ingenuo sería *«el mes de mayor producción»*. Y está mal, porque el mes de
mayor producción es casi siempre **un mes con suerte**: un pozo nuevo que entró, un mes
sin paradas. Veámoslo con un campo inventado.

In [ ]:
# un campo de mentira: sube, se queda en meseta, y baja. Con ruido, como la vida.
rng = np.random.default_rng(0)
meses = np.arange(200)

# sube 3 anios, se queda 6 en meseta, y despues declina
forma = 100 * np.minimum(1, meses / 36) * np.exp(-np.maximum(0, meses - 108) / 60)
ruido = rng.normal(1, 0.12, len(meses))     # el ruido de siempre en campo
falso = pd.Series(forma * ruido)

suavizado = falso.rolling(6, center=True, min_periods=1).mean()

pico_crudo = int(falso.idxmax())        # el mes de mayor produccion
pico_suave = int(suavizado.idxmax())    # el mes de mayor produccion SUAVIZADA

plt.figure(figsize=(10, 3.6))
plt.plot(meses / 12, falso, color="#E5E7EB", lw=1.2, label="mes a mes")
plt.plot(meses / 12, suavizado, color="#2D2D2D", lw=2, label="suavizado a 6 meses")
plt.plot(pico_crudo / 12, falso[pico_crudo], "o", ms=12, mfc="none",
         mec="#C82B40", mew=2.5, label="maximo del dato crudo")
plt.plot(pico_suave / 12, suavizado[pico_suave], "*", ms=18, color="#16A34A",
         label="el pico que usamos")
plt.xlabel("anios")
plt.ylabel("produccion")
plt.legend(fontsize=8)
plt.title("Con meseta, el maximo crudo cae en cualquier lado")
plt.show()

print("el maximo crudo cae en el mes  ", pico_crudo)
print("el maximo suavizado, en el mes ", pico_suave)
print("se separan", abs(pico_crudo - pico_suave), "meses")

**Lectura**: los dos criterios dan meses distintos, y eso corre el cero del reloj.

En los 66 campos del archivo original del regulador esto se midió:

| | |
|---|---|
| El máximo crudo y el suavizado coinciden en | **2 de 66** campos |
| Se separan más de medio año en | **15** campos (hasta **58 meses** en Statfjord) |
| No tienen pico sino **meseta** de más de un año | **11 de 66** |
| **Vuelven a subir** 3+ años después (pozos nuevos) | **7 de 66** |

> ⚠️ **Lo que hay que llevarse**: en esos casos «el pico» **no es un dato, es una
> decisión**. Nosotros elegimos el máximo de la producción suavizada a seis meses.
> Es defendible, no es la única opción, y está escrita en `preparar_datos.py` para que
> cualquiera pueda discutirla.

> 🔧 **Mini-ejercicio 1**: cambien la ventana de suavizado de 6 a 3 y a 12 meses en el
> campo de mentira. ¿Cuánto se mueve el pico? ¿Qué ventana les parece más defendible?

In [ ]:
# Escribe tu solucion aqui

> 📌 **Decisión 1**: alineamos todo desde el pico. Ya viene hecho en la columna
> `mes_desde_pico`, y el criterio está en `preparar_datos.py`: se busca el máximo de la
> producción **suavizada a seis meses**, para que un solo mes bueno no se lleve el pico
> a un lugar equivocado.

## 1.5 · ¿Todos los campos declinan igual?

Cada campo tiene un tamaño distinto. Pero, ¿tienen la misma **forma**?

In [ ]:
# guardamos cada campo como una serie suelta: nos va a servir todo el cuaderno
campos = {}
for nombre, g in d.groupby("campo"):
    g = g.sort_values("mes_desde_pico")
    campos[nombre] = g.oil_bpd.reset_index(drop=True)

print("campos guardados:", len(campos))
print("ejemplo, los primeros 5 meses de DRAUGEN:")
print(campos["DRAUGEN"].head().round(0).tolist())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8))
grandes = ["STATFJORD", "GULLFAKS", "EKOFISK", "DRAUGEN", "NORNE"]

for nombre in grandes:
    q = campos[nombre].iloc[:25 * 12]                 # 25 anios
    t = np.arange(len(q)) / 12
    # izquierda: en barriles
    ejes[0].plot(t, q / 1000, lw=1.5, label=nombre.title())
    # derecha: como % de su propio pico, y en escala logaritmica
    ejes[1].plot(t, 100 * q / q.iloc[:6].mean(), lw=1.5)

ejes[0].set_ylabel("miles de bbl/dia")
ejes[0].set_title("En barriles: cada campo es un mundo")
ejes[0].legend(fontsize=8)
ejes[1].set_yscale("log")
ejes[1].set_ylabel("% de su propio pico")
ejes[1].set_title("En % del pico y escala log: casi la misma curva")
for e in ejes:
    e.set_xlabel("anios desde el pico")
plt.tight_layout()
plt.show()

**Lectura**: es el mismo truco de la Clase 2 — **comparar cada activo consigo mismo**.
En barriles, Statfjord y Norne no tienen nada que ver. En porcentaje de su propio pico,
son casi la misma curva bajando.

Y fíjense en la escala logarítmica de la derecha: las curvas se ven **casi rectas**.
Guarden eso, porque es toda la sección 2.

> 📌 **Decisión 2**: vamos a modelar la **forma**, no el tamaño. El tamaño lo pone
> cada campo con su propio nivel de arranque.

## 1.6 · ¿Qué es «el acumulado», y por qué es lo que se firma?

La producción diaria interesa al de operaciones. Al comité de reservas le interesa
**el total**: cuántos barriles van a salir en total. Eso es el **acumulado**.

In [ ]:
# el acumulado es sumar la produccion mes a mes.
# Ojo: oil_bpd es un CAUDAL (barriles por dia), asi que para pasar a
# volumen del mes hay que multiplicar por los dias de ese mes.
g = d[d.campo == "DRAUGEN"].sort_values("mes_desde_pico")

volumen_mensual = g.oil_bpd * g.dias        # barriles en cada mes
acumulado = volumen_mensual.cumsum()        # cumsum = suma que se va acumulando

plt.figure(figsize=(10, 3.4))
plt.plot(g.mes_desde_pico / 12, acumulado / 1e6, color="#C82B40", lw=2)
plt.xlabel("anios desde el pico")
plt.ylabel("acumulado\n[millones de barriles]")
plt.title("Campo DRAUGEN: lo que lleva producido desde el pico")
plt.show()

print("a los 5 anios: ", round(acumulado.iloc[59] / 1e6, 1), "MMbbl")
print("a los 12 anios:", round(acumulado.iloc[143] / 1e6, 1), "MMbbl")

**Lectura**: la curva se va **aplanando**. Cada año agrega menos que el anterior,
porque el caudal cae. Eso es justamente lo que hace difícil la pregunta: la mayor parte
de lo que falta está en una cola larga y baja.

> 🔧 **Mini-ejercicio 1**: ¿qué fracción del acumulado a 12 años ya estaba producida
> a los 5 años? Calcúlenlo para Draugen.

In [ ]:
# Escribe tu solucion aqui

## 1.7 · El encargo, dibujado

Vamos a tapar todo lo posterior al año 5. Eso es lo que se ve al momento de firmar.

In [ ]:
# a partir de aca, estas dos constantes gobiernan todo el cuaderno
AJUSTE = 5 * 12        # con cuanta historia ajustamos: 5 anios = 60 meses
OBJETIVO = 12 * 12     # que queremos predecir: el acumulado a 12 anios

q = campos["DRAUGEN"]

plt.figure(figsize=(10, 3.8))
plt.plot(np.arange(AJUSTE) / 12, q.iloc[:AJUSTE] / 1000, color="#C82B40", lw=2)
plt.axvspan(AJUSTE / 12, 12, color="#E5E7EB")          # la mancha gris
plt.text(8.5, q.max() / 2000, "esto todavia\nno paso", ha="center",
         fontsize=13, color="#6B7280", fontweight="bold")
plt.axvline(AJUSTE / 12, color="#6B1525", ls="--", lw=1.8)
plt.xlim(0, 12)
plt.xlabel("anios desde el pico")
plt.ylabel("miles de bbl/dia")
plt.title("Campo DRAUGEN: cinco anios de historia. Cuanto produce en los proximos siete?")
plt.show()

## 1.8 · Cierre de la exploración

| Lo que vimos | Lo que decidimos |
|---|---|
| Ekofisk arrancó en 1971 y Grane en 2003 | Alinear todo **desde el pico**, no por calendario |
| En % del pico, todos los campos se parecen | Modelar la **forma**, no el tamaño |
| En escala logarítmica las curvas son casi rectas | Empezar por una **recta sobre el logaritmo** |
| El acumulado se aplana: la cola pesa | El error va a estar **en la cola** |
| 54 campos ya terminaron sus 12 años | Podemos **verificar** el pronóstico, no solo hacerlo |

Recién ahora se puede modelar.

---

# 2 · Arps exponencial: la recta de la Clase 1, con nombre

**J. J. Arps**, en 1945, miró muchísimas curvas de producción reales y buscó una familia
de fórmulas que las describiera. No dedujo nada de la física: **ajustó lo que veía**.
Ochenta años después sigue siendo la base de cómo se declaran las reservas.

La más simple de esa familia es la **exponencial**: el campo pierde **el mismo
porcentaje** cada mes. Y cuando algo pierde el mismo porcentaje siempre, al dibujar su
logaritmo sale una **recta**.

## 2.1 · ¿Qué quiere decir «ajustar»?

Antes de escribir `polyfit`, hay que saber qué le estamos pidiendo. Y no es magia.

Ajustar es **elegir los números que hacen el error más chico**. El error de una recta es:

1. para cada mes, la distancia entre el dato y la recta (lo que le *sobra* a la recta),
2. cada una de esas distancias **elevada al cuadrado**,
3. todo sumado.

Se elevan al cuadrado por una razón de ingeniería: **un error grande no se compensa con
dos aciertos.** Elevar al cuadrado hace que equivocarse mucho una vez cueste más que
equivocarse poco muchas veces.

In [ ]:
q = campos["DRAUGEN"]
y = q.iloc[:AJUSTE].values
t = np.arange(AJUSTE)


def error_de(pendiente, altura, y, t):
    """El error total de UNA recta: la suma de lo que le sobra, al cuadrado."""
    recta = pendiente * t + altura
    sobra = y - recta
    return (sobra ** 2).sum()


# probemos tres rectas a ojo y veamos cual es "menos mala"
for pend, alt in [(-3000, 240000), (-2400, 215000), (-2000, 200000)]:
    e = error_de(pend, alt, y, t)
    print(f"pendiente {pend:>6} , altura {alt:>7}  ->  error {e:.3e}")

> 🔧 **Mini-ejercicio 2**: prueben a mano otras tres rectas y vean si consiguen bajar
> el error de la mejor de arriba. ¿Cuántos intentos les lleva?

In [ ]:
# Escribe tu solucion aqui

## 2.2 · `polyfit` no busca: **despeja**

Acá está lo que hace especial a la recta.

Para una recta, el punto donde el error es mínimo **se puede escribir con una fórmula**.
No hay que probar, ni buscar, ni empezar por ningún lado: se despeja, igual que uno
despeja *x* en una ecuación de la escuela.

Vamos a resolverla a mano y comparar con `polyfit`.

In [ ]:
ly = np.log(y)          # trabajamos sobre el logaritmo, como vimos

# LA FORMULA, escrita a mano. Es la misma de siempre:
#   pendiente = (promedio de t*y  -  promedio de t * promedio de y)
#               ---------------------------------------------------
#               (promedio de t*t  -  promedio de t al cuadrado)
pendiente = ((t * ly).mean() - t.mean() * ly.mean()) / \
            ((t * t).mean() - t.mean() ** 2)
altura = ly.mean() - pendiente * t.mean()

print("a mano :", round(pendiente, 6), round(altura, 4))
print("polyfit:", np.polyfit(t, ly, 1).round(6))

**Lectura**: dan **exactamente lo mismo**, hasta el último decimal.

`polyfit` es esa fórmula, nada más. Por eso:

- es **instantáneo** — no prueba nada, despeja;
- da **siempre la misma respuesta**, se corra donde se corra;
- **no necesita** que uno le diga por dónde empezar ni entre qué límites buscar.

Guarden esas tres propiedades, porque la hiperbólica **no tiene ninguna de las tres**.

## 2.3 · Cuando no hay fórmula: hay que caminar

La hiperbólica de Arps es

$$q(t) = \\frac{q_i}{(1 + b\\,D_i\\,t)^{1/b}}$$

y el problema es **dónde está b**: en el exponente, y además dentro del paréntesis.
No se puede despejar. No existe fórmula para el mínimo.

Entonces la computadora hace lo único que puede: **empieza en algún lado y camina cuesta
abajo**. Mira el error donde está, mira hacia dónde baja, da un paso, y repite hasta que
ya no puede bajar más.

Eso es `curve_fit`. Y por eso pide dos cosas que `polyfit` no pide:

- **`p0`** — por dónde empezar a caminar;
- **`bounds`** — barandas, para que no se caiga por un barranco sin sentido físico.

In [ ]:
from scipy.optimize import least_squares


def hiperbolica(t, qi, Di, b):
    """Arps (1945). Con b -> 0 se vuelve la exponencial."""
    return qi / (1 + b * Di * t) ** (1 / b)


# vamos a espiar el camino: cada vez que la computadora prueba
# una combinacion, la anotamos
camino = []


def lo_que_sobra(parametros):
    Di, b = parametros
    camino.append((Di, b))
    return y - hiperbolica(t, y[0], Di, b)


resultado = least_squares(lo_que_sobra, x0=[0.006, 0.10],
                          bounds=([1e-4, 1e-3], [0.08, 1.8]))

print("empezo en   Di =", 0.006, " b =", 0.10)
print("termino en  Di =", round(resultado.x[0], 5), " b =", round(resultado.x[1], 3))
print("probo", len(camino), "combinaciones para llegar")

**Lectura**: fíjense en el número: la computadora **probó 27 combinaciones** antes de
quedarse con una. No la despejó: la buscó.

Y ojo con una consecuencia práctica: como `b` y `Di` aparecen multiplicados dentro del
paréntesis, **se compensan** — un `Di` más alto con un `b` más alto da casi la misma
curva. Con solo cinco años de datos el ajuste no los distingue del todo.

> Por eso dos ingenieros pueden ajustar el mismo campo, reportar `b` distintos, y
> **ninguno estar equivocado**.

In [ ]:
# comprobemoslo: dos combinaciones muy distintas, ¿cuanto error da cada una?
for Di, b in [(0.020, 0.50), (0.040, 1.20), (resultado.x[0], resultado.x[1])]:
    e = ((y - hiperbolica(t, y[0], Di, b)) ** 2).sum()
    print(f"Di = {Di:.3f} , b = {b:.2f}  ->  error {e:.3e}")

## 2.4 · Y ahora sí: qué es **b**, en una frase

`b` **no cambia cuánto declina el campo hoy. Cambia cuánto va a declinar mañana.**

La declinación de un campo hiperbólico, año a año, es:

$$D(t) = \\frac{D_i}{1 + b\\,D_i\\,t}$$

Miren el denominador: crece con el tiempo, así que **la declinación se va achicando**.
Y cuánto se achica lo decide `b`.

In [ ]:
t_anios = np.arange(0, 20 * 12)
Di = 0.02

plt.figure(figsize=(9, 3.6))
for b, color, etiqueta in [(0.001, "#2D2D2D", "b = 0   (exponencial)"),
                           (0.5, "#2563EB", "b = 0.5 (lo habitual)"),
                           (1.0, "#16A34A", "b = 1   (armonica)")]:
    # la declinacion de ESE anio, en porcentaje
    D_del_anio = 100 * (1 - np.exp(-12 * Di / (1 + b * Di * t_anios)))
    plt.plot(t_anios / 12, D_del_anio, color=color, lw=2.2, label=etiqueta)

plt.xlabel("anios")
plt.ylabel("cuanto declina ESE anio [%]")
plt.ylim(0, 26)
plt.legend()
plt.title("Esto es lo que b controla")
plt.show()

print("con b = 0   la caida es del 21 % todos los anios, para siempre")
print("con b = 1   arranca en 21 % y a los 20 anios ya es del 4 %")

**Lectura andragógica**: ustedes ya conocen `b` sin haberlo llamado así.

Es la diferencia entre el pozo que **se muere rápido y se acabó**, y el que
**«se queda goteando para siempre»** — ese que nadie termina de abandonar porque todos
los años sigue dando un poco.

Físicamente: `b` mide **qué tan parejo es el yacimiento**. Una roca homogénea se drena de
forma pareja y da `b` cerca de 0. Una roca con zonas buenas y zonas apretadas primero
entrega lo fácil y después lo lento — y eso da `b` alto.

> ⚠️ **Y una baranda que no se negocia**: si el ajuste da `b > 1`, el acumulado que
> predice la fórmula es **infinito**. Ningún campo produce infinito. Si les sale eso,
> el ajuste está mal, no el campo.

In [ ]:
q = campos["DRAUGEN"]

y = q.iloc[:AJUSTE].values       # los primeros 5 anios, en bbl/dia
t = np.arange(AJUSTE)            # 0, 1, 2, ... 59  (meses)

# ajustamos una RECTA sobre el LOGARITMO de la produccion.
# polyfit con grado 1 devuelve [pendiente, altura]
b_exp = np.polyfit(t, np.log(y), 1)

print("pendiente (por mes):", round(b_exp[0], 4))

# la pendiente es -D mensual. La pasamos a porcentaje anual:
D_anual = 100 * (1 - np.exp(b_exp[0] * 12))
print("este campo declina", round(D_anual), "% al anio")

In [ ]:
# lo dibujamos en los dos ejes, para ver por que se llama "exponencial"
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.6))
for eje, log in zip(ejes, [False, True]):
    eje.plot(t / 12, y / 1000, "o", ms=2.5, color="#9CA3AF", label="lo que midio el campo")
    eje.plot(t / 12, np.exp(np.polyval(b_exp, t)) / 1000, color="#C82B40", lw=2.2,
             label="la recta de Arps")
    if log:
        eje.set_yscale("log")
        eje.set_title("En escala logaritmica: es una RECTA")
    else:
        eje.set_title("En escala normal: es una curva que cae")
    eje.set_xlabel("anios desde el pico")
    eje.set_ylabel("miles de bbl/dia")
ejes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Veámoslo pasar: doblando el eje de a poco

Decir *«en escala logarítmica es una recta»* y mostrarlo son dos cosas distintas.
Vamos a **doblar el eje de forma continua** y mirar qué le pasa a la curva.

La herramienta es una familia de transformaciones con una perilla, **λ**:

$$z = \frac{y^{\lambda} - 1}{\lambda} \qquad\text{y cuando } \lambda \to 0,\ \ z \to \log(y)$$

- **λ = 1** → el dato como sale, escala normal
- **λ = 0** → el logaritmo

Abajo, en cada instante, ajustamos una **recta** y dibujamos **lo que le sobra** a esa
recta. Si el residuo tiene forma de arco, la recta era el modelo equivocado.

In [ ]:
from matplotlib import animation
from IPython.display import HTML


def doblar(y, lam):
    """Con lam=1 devuelve el dato como esta; con lam=0, su logaritmo."""
    if abs(lam) < 1e-9:
        return np.log(y)
    return (y ** lam - 1) / lam


y_anim = campos["DRAUGEN"].iloc[:AJUSTE].values
t_anim = np.arange(len(y_anim))

# la secuencia de la perilla: arranca quieta en 1, baja a 0, y se queda
LAMBDAS = np.concatenate([np.ones(4), np.linspace(1, 0, 26), np.zeros(6)])

plt.rcParams["figure.dpi"] = 76
fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(7.6, 4.9), sharex=True,
                                    gridspec_kw=dict(height_ratios=[2.1, 1]))
fig.subplots_adjust(hspace=0.14, top=0.88)


def marco(i):
    lam = LAMBDAS[i]
    arriba.clear(); abajo.clear()

    # doblamos el eje y normalizamos, para poder comparar formas
    z = doblar(y_anim, lam)
    z = (z - z.mean()) / z.std()

    # la mejor RECTA sobre el dato doblado
    tt = np.linspace(-1, 1, len(z))
    recta = np.polyval(np.polyfit(tt, z, 1), tt)

    # lo que le sobra a la recta, y cuanto se arquea
    resto = z - recta
    arco = np.polyval(np.polyfit(tt, resto, 2), tt)
    curvatura = abs(np.polyfit(tt, resto, 2)[0])

    arriba.plot(t_anim / 12, z, "o", ms=4, color="#9CA3AF")
    arriba.plot(t_anim / 12, recta, color="#C82B40", lw=2.6)
    arriba.set_ylabel("produccion\n(eje doblado)", fontsize=9.5)
    arriba.set_ylim(-2.6, 2.6); arriba.grid(alpha=0.3)
    nombre = ("escala NORMAL" if lam > 0.97
              else "escala LOGARITMICA" if lam < 0.03 else "doblando el eje...")
    arriba.set_title(f"lambda = {lam:.2f}   ·   {nombre}", fontsize=13,
                     fontweight="bold", color="#6B1525", loc="left")

    abajo.axhline(0, color="#2D2D2D", lw=1.4)
    abajo.plot(t_anim / 12, resto, "o", ms=3, color="#9CA3AF")
    abajo.fill_between(t_anim / 12, 0, arco, color="#C82B40", alpha=0.35)
    abajo.plot(t_anim / 12, arco, color="#C82B40", lw=2)
    abajo.set_ylim(-1.5, 1.5); abajo.grid(alpha=0.3)
    abajo.set_xlabel("anios desde el pico", fontsize=9.5)
    abajo.set_ylabel("lo que le sobra\na la recta", fontsize=9.5)
    abajo.text(0.985, 0.06, f"curvatura: {curvatura:.2f}", transform=abajo.transAxes,
               ha="right", fontsize=12, fontweight="bold",
               color="#16A34A" if curvatura < 0.2 else "#C82B40")
    return []


ani = animation.FuncAnimation(fig, marco, frames=len(LAMBDAS),
                              interval=110, blit=False)
plt.close()                       # para que no salga el grafico suelto
HTML(ani.to_jshtml(default_mode="once"))

**Lectura**: miren el panel de abajo, no el de arriba.

Con λ = 1 el residuo tiene una **panza**: la recta pasa por encima al principio y al
final, y por debajo en el medio. Eso es la firma de haber ajustado una recta a algo que
no lo era. La curvatura vale **0,50**.

A medida que λ baja, la panza se aplana. En λ = 0 el residuo queda **repartido a los dos
lados del cero, sin forma**, y la curvatura cae a **0,06** — ocho veces menos.

**Y eso es toda la utilidad**: cuando el residuo no tiene forma, la recta es el modelo
correcto. Y una recta se ajusta con `polyfit(..., 1)`, tiene **dos** números, y uno de
ellos —la pendiente— **es la tasa de declinación D**.

> 🤔 **Pregunta clave**: ¿por qué normalizamos `z` en cada cuadro
> (`(z - z.mean()) / z.std()`)? ¿Qué pasaría si no lo hiciéramos?

### ¿Y funciona siempre?

Draugen quedó precioso. Pero la regla de la casa es no creerle a un campo.

In [ ]:
def curvatura_del_residuo(y, lam):
    """Cuanto se arquea lo que le sobra a una recta. 0 = perfectamente recta."""
    z = doblar(y, lam)
    z = (z - z.mean()) / z.std()
    tt = np.linspace(-1, 1, len(z))
    resto = z - np.polyval(np.polyfit(tt, z, 1), tt)
    return abs(np.polyfit(tt, resto, 2)[0])


mejora = 0
total = 0
for nombre, q in campos.items():
    yy = q.iloc[:AJUSTE].values
    yy = yy[yy > 0]
    if len(yy) < 48:
        continue
    total = total + 1
    if curvatura_del_residuo(yy, 0.0) < curvatura_del_residuo(yy, 1.0):
        mejora = mejora + 1

print("el logaritmo endereza la curva en", mejora, "de", total, "campos")
print("o sea el", round(100 * mejora / total), "%")

**Lectura — y acá está el puente al resto de la clase**:

Funciona en **2 de cada 3 campos**. En el tercio restante, doblar el eje **no alcanza**:
la curva sigue arqueada aunque la miremos en logaritmo.

Ese tercio son los campos con **cola** — los que se resisten a morir. Y son exactamente
los que la exponencial va a subestimar.

Guarden eso, porque es la razón de que exista la sección 4.

### Y ahora proyectamos

In [ ]:
# np.polyval evalua la recta en los meses que le pidamos,
# y np.exp deshace el logaritmo para volver a barriles por dia
pron_exp = np.exp(np.polyval(b_exp, np.arange(OBJETIVO)))

# el acumulado: caudal por dias. Usamos 30.4 dias, el promedio de un mes
acum_exp = pron_exp.sum() * 30.4

# y lo que DE VERDAD paso (esto es hacer trampa, pero es una auditoria)
g = d[d.campo == "DRAUGEN"].sort_values("mes_desde_pico")
acum_real = (g.oil_bpd * g.dias).iloc[:OBJETIVO].sum()

print("Arps exponencial predice:", round(acum_exp / 1e6, 1), "MMbbl")
print("lo que de verdad paso:   ", round(acum_real / 1e6, 1), "MMbbl")
print("error:", round(100 * (acum_exp / acum_real - 1)), "%")

> 🤔 **Pregunta clave**: se equivocó por poco. ¿Alcanza con un campo para decir que
> el método es bueno?

---

# 3 · El defecto: lo medimos en los 54 campos

Un campo no prueba nada. Vamos a repetir exactamente lo mismo en los 54, y mirar
**cómo se distribuyen los errores**.

In [ ]:
def acumulado_real(nombre, meses):
    """Cuantos barriles produjo de verdad ese campo en esos meses."""
    g = d[d.campo == nombre].sort_values("mes_desde_pico")
    return (g.oil_bpd * g.dias).iloc[:meses].sum()


def ajustar_exponencial(q):
    """Devuelve los coeficientes de la recta sobre el logaritmo."""
    y = q.iloc[:AJUSTE].values
    t = np.arange(AJUSTE)
    m = y > 0                             # el log de 0 no existe
    if m.sum() < 24:
        return None
    b = np.polyfit(t[m], np.log(y[m]), 1)
    if b[0] >= 0:                         # si "declina" hacia arriba, no sirve
        return None
    return b

print("funciones listas")

In [ ]:
# recorremos los 54 campos, uno por uno
filas = []
for nombre, q in campos.items():
    b = ajustar_exponencial(q)
    if b is None:
        continue

    pred = np.exp(np.polyval(b, np.arange(OBJETIVO))).sum() * 30.4
    real = acumulado_real(nombre, OBJETIVO)

    filas.append({
        "campo": nombre,
        "real": real,
        "exp": pred,
        "D_anual": 100 * (1 - np.exp(b[0] * 12)),
        "acum5": acumulado_real(nombre, AJUSTE),
    })

r = pd.DataFrame(filas)
r["e_exp"] = 100 * (r.exp / r.real - 1)      # error en porcentaje

print(len(r), "campos evaluados")
r[["campo", "D_anual", "e_exp"]].head(8).round(1)

In [ ]:
print("Arps exponencial, en los", len(r), "campos:")
print("  error tipico (mediana del |error|):", round(r.e_exp.abs().median()), "%")
print("  SESGO       (mediana del error):   ", round(r.e_exp.median()), "%")
print("  peor caso:                         ", round(r.e_exp.abs().max()), "%")
print()
print("  declinacion anual D: mediana", round(r.D_anual.median()), "%/anio")

In [ ]:
plt.figure(figsize=(10, 3.6))
plt.hist(r.e_exp, bins=np.arange(-60, 61, 6), color="#C82B40", alpha=0.7)
plt.axvline(0, color="#2D2D2D", lw=2)                       # acertar es aca
plt.axvline(r.e_exp.median(), color="#C82B40", lw=2.4, ls="--")   # donde cae de verdad
plt.xlabel("error del acumulado a 12 anios [%]   (negativo = predijo de menos)")
plt.ylabel("cantidad de campos")
plt.title("El error NO esta centrado en cero")
plt.show()

**Lectura — y esta es la lámina que justifica el resto de la clase**:

El histograma está corrido a la izquierda. La exponencial **no se equivoca al azar**:
se equivoca **siempre para el mismo lado**, prediciendo de menos.

### Error y sesgo no son lo mismo

| | qué es | por qué importa |
|---|---|---|
| **Error** (imprecisión) | Le pego a veces arriba y a veces abajo | **Se compensa**: con veinte campos, los de más y los de menos se cancelan |
| **Sesgo** (error sistemático) | Le pego **siempre** para el mismo lado | **No se compensa nunca**: con veinte campos me equivoco veinte veces en la misma dirección |

Un sesgo de −7 % suena conservador y prudente. Pero significa que se **subvalúan activos
de forma sistemática**, y que se abandonan campos antes de tiempo.

Y tiene una explicación física, no estadística: **la exponencial no cree en la cola.**

---

# 4 · La hiperbólica: qué es b

Un campo no es un tanque homogéneo. Tiene zonas muy permeables que se drenan rápido, y
zonas apretadas que entregan lento. Al principio sale lo fácil y la caída es rápida;
cuando lo fácil se agotó, queda lo lento — y **lo lento cae despacio**.

Por eso la declinación **se frena**. Arps lo metió en la fórmula con un solo botón, **b**:

$$q(t) = \frac{q_i}{(1 + b \, D_i \, t)^{1/b}}$$

- **b = 0** → exponencial (la de recién)
- **0 < b < 1** → hiperbólica: lo que se ve casi siempre
- **b = 1** → armónica: la cola más gorda posible

In [ ]:
# la formula, tal cual la escribio Arps en 1945
def hiperbolica(t, qi, Di, b):
    return qi / (1 + b * Di * t) ** (1 / b)


# veamos que hace b, con numeros inventados
t_demo = np.arange(0, 144)
plt.figure(figsize=(10, 3.4))
for b, color, etiqueta in [(0.001, "#2D2D2D", "b = 0   (exponencial)"),
                           (0.5, "#2563EB", "b = 0.5 (lo habitual)"),
                           (1.0, "#16A34A", "b = 1   (armonica)")]:
    plt.plot(t_demo / 12, hiperbolica(t_demo, 100, 0.02, b), lw=2, color=color,
             label=etiqueta)
plt.xlabel("anios")
plt.ylabel("% del inicio")
plt.title("Todas arrancan igual. Se separan en la COLA.")
plt.legend()
plt.show()

**Lectura**: las tres curvas son casi idénticas los primeros años — que es todo lo que
se ve al firmar. Se separan justo donde nadie puede mirar.

In [ ]:
def ajustar_hiperbolica(q):
    """Busca los tres numeros (qi, Di, b) que mejor ajustan."""
    y = q.iloc[:AJUSTE].values
    t = np.arange(AJUSTE)
    m = y > 0
    if m.sum() < 24:
        return None
    try:
        # p0 es por donde empieza a buscar.
        # bounds son las BARANDAS: sin ellas se va a valores sin sentido fisico.
        p, _ = curve_fit(hiperbolica, t[m], y[m],
                         p0=[y[m][0], 0.02, 0.5],
                         bounds=([0, 1e-6, 1e-3], [np.inf, 1.0, 2.0]),
                         maxfev=20000)
        return p
    except Exception:
        return None


p = ajustar_hiperbolica(campos["DRAUGEN"])
print("qi =", round(p[0]), "bbl/dia")
print("Di =", round(p[1], 4), "por mes")
print("b  =", round(p[2], 2), " <- este campo tiene MUCHA cola")

In [ ]:
# las dos curvas sobre el mismo campo, y lo que de verdad paso
q = campos["DRAUGEN"]
t_full = np.arange(OBJETIVO)

plt.figure(figsize=(10, 4))
plt.axvspan(0, AJUSTE / 12, color="#EEF2FF")               # la zona que se vio
plt.plot(t_full / 12, q.iloc[:OBJETIVO] / 1000, color="#2D2D2D", lw=1.8,
         label="lo que de verdad produjo")
plt.plot(t_full / 12, np.exp(np.polyval(b_exp, t_full)) / 1000, "--",
         color="#C82B40", lw=2, label="Arps exponencial")
plt.plot(t_full / 12, hiperbolica(t_full, *p) / 1000, "-.",
         color="#16A34A", lw=2, label=f"Arps hiperbolica (b={p[2]:.2f})")
plt.axvline(AJUSTE / 12, color="#6B1525", ls=":", lw=1.6)
plt.xlabel("anios desde el pico")
plt.ylabel("miles de bbl/dia")
plt.legend()
plt.title("Las dos se ajustan igual de bien a lo que se vio. Se separan despues.")
plt.show()

In [ ]:
# ahora la hiperbolica en los 54 campos
hip = []
for nombre in r.campo:
    p = ajustar_hiperbolica(campos[nombre])
    if p is None:
        hip.append({"hip": np.nan, "b": np.nan})
    else:
        hip.append({"hip": hiperbolica(np.arange(OBJETIVO), *p).sum() * 30.4,
                    "b": p[2]})

h = pd.DataFrame(hip, index=r.index)
r["hip"], r["b"] = h.hip, h.b
r["e_hip"] = 100 * (r.hip / r.real - 1)

print("Arps HIPERBOLICA:")
print("  error tipico:", round(r.e_hip.abs().median()), "%")
print("  SESGO:       ", round(r.e_hip.median()), "%")
print("  peor caso:   ", round(r.e_hip.abs().max()), "%")
print()
print("  b: mediana", round(r.b.median(), 2),
      "| campos con b > 0.1:", int((r.b > 0.1).sum()), "de", len(r))

**Lectura**: la hiperbólica **se ganó su lugar**. Mata el sesgo (de −7 % a +1 %) y baja
el peor caso de forma clara.

Y el dato del exponente es interesante: la mediana de b es muy chica, o sea que en la
mitad de los campos **la hiperbólica eligió sola volverse casi exponencial**. En la otra
mitad, no — y ahí es donde importó.

> 🔧 **Mini-ejercicio 2**: hagan un gráfico de `b` contra `e_exp`. ¿Los campos donde la
> exponencial falló más son los que tienen b más grande?

In [ ]:
# Escribe tu solucion aqui

---

# 5 · Y ahora, en dólares

Todo lo que hicimos pronostica **barriles**. Pero nadie produce barriles que cuesten más
de lo que valen.

Un campo tiene dos costos:

- **Variable** — lo que cuesta sacar cada barril (energía, químicos, tratar el agua).
  Se paga por barril.
- **Fijo** — lo que cuesta tener el campo **abierto** (plataforma, gente, mantenimiento).
  Se paga igual, produzca mucho o poco.

Cuando la producción baja tanto que ya no alcanza para pagar el costo fijo, el campo
**pierde plata todos los meses**. Ese caudal es el **límite económico**:

$$q_{límite} = \frac{\text{costo fijo del mes}}{(\text{precio} - \text{costo variable}) \times \text{días}}$$

In [ ]:
# Estos tres numeros los pone finanzas, no el ingeniero de yacimientos.
# Van a la vista para que se puedan discutir y cambiar.
PRECIO = 70.0        # USD por barril
OPEX = 15.0          # USD por barril producido
COSTO_FIJO = 8e6     # USD al mes que cuesta tener el campo abierto

MARGEN = PRECIO - OPEX
Q_LIMITE = COSTO_FIJO / (MARGEN * 30.4)

print("margen neto:", MARGEN, "USD/bbl")
print("limite economico:", round(Q_LIMITE), "bbl/dia")
print()
print("Por debajo de ese caudal, el campo pierde plata todos los meses.")

> 🤔 **Pregunta clave**: si el precio del crudo baja de 70 a 50 USD/bbl, ¿el límite
> económico sube o baja? ¿Y qué le pasa a la fecha de abandono?

In [ ]:
def mes_de_abandono(curva):
    """Primer mes en que la curva cae por debajo del limite economico."""
    debajo = np.where(curva < Q_LIMITE)[0]
    if len(debajo) == 0:
        return None                       # nunca cruza
    return int(debajo[0])


# proyectamos 60 anios para los dos ajustes de Draugen
N = 60 * 12
curva_exp = np.exp(np.polyval(b_exp, np.arange(N)))
p_hip = ajustar_hiperbolica(campos["DRAUGEN"])
curva_hip = hiperbolica(np.arange(N), *p_hip)

m_exp = mes_de_abandono(curva_exp)
m_hip = mes_de_abandono(curva_hip)

print("segun la EXPONENCIAL, Draugen cierra a los", round(m_exp / 12), "anios")
print("segun la HIPERBOLICA:", "NUNCA cierra" if m_hip is None
      else f"a los {round(m_hip/12)} anios")

**Lectura**: la hiperbólica dice que el campo **no se muere nunca**.

Con `b` cerca de 1 la declinación se frena tanto que la curva jamás cruza el límite.
El modelo está afirmando algo físicamente imposible.

### La corrección estándar: declinación terminal

Se le pone un **piso a la declinación**: cuando la hiperbólica cae por debajo de un
mínimo — habitualmente **5 % al año** — se cambia a exponencial y de ahí en adelante
decae parejo.

In [ ]:
D_MIN = 0.05        # 5 % al anio: el piso de la declinacion


def hiperbolica_con_freno(p, n, d_min=D_MIN):
    """Hiperbolica que cambia a exponencial cuando declina demasiado poco."""
    qi, Di, b = p
    t = np.arange(n)
    q = hiperbolica(t, *p)

    # la declinacion instantanea de la hiperbolica en cada mes
    D_inst = Di / (1 + b * Di * t)

    # el minimo, pasado de "por anio" a "por mes"
    d_min_mes = -np.log(1 - d_min) / 12

    # a partir del primer mes en que declina demasiado poco, exponencial
    lentos = np.where(D_inst <= d_min_mes)[0]
    if len(lentos) > 0:
        i = lentos[0]
        q[i:] = q[i] * np.exp(-d_min_mes * (t[i:] - t[i]))
    return q


curva_ter = hiperbolica_con_freno(p_hip, N)
m_ter = mes_de_abandono(curva_ter)
print("con declinacion terminal, Draugen cierra a los", round(m_ter / 12), "anios")

In [ ]:
# las tres curvas y el limite, en escala logaritmica
plt.figure(figsize=(10, 4.2))
t = np.arange(N) / 12
plt.plot(t, curva_exp, "--", color="#C82B40", lw=2, label="Arps exponencial")
plt.plot(t, curva_hip, ":", color="#9CA3AF", lw=2, label="hiperbolica sin freno")
plt.plot(t, curva_ter, color="#16A34A", lw=2.2, label="hiperbolica + terminal")
plt.axhline(Q_LIMITE, color="#2D2D2D", lw=2)
plt.axhspan(0, Q_LIMITE, color="#FEE2E2")
plt.text(0.5, Q_LIMITE * 0.45, "aca el campo PIERDE plata",
         color="#C82B40", fontweight="bold")
plt.yscale("log")
plt.ylim(Q_LIMITE * 0.25, campos["DRAUGEN"].max() * 1.4)
plt.xlim(0, 60)
plt.xlabel("anios desde el pico")
plt.ylabel("bbl/dia")
plt.legend()
plt.title("La curva elegida decide en que anio se cierra el campo")
plt.show()

In [ ]:
# ¿en cuantos campos pasa esto? Lo contamos en los 54.
sin_freno, con_freno = 0, 0
for nombre in r.campo:
    pp = ajustar_hiperbolica(campos[nombre])
    if pp is None:
        continue
    if mes_de_abandono(hiperbolica(np.arange(N), *pp)) is None:
        sin_freno = sin_freno + 1
    if mes_de_abandono(hiperbolica_con_freno(pp, N)) is None:
        con_freno = con_freno + 1

print("campos que la hiperbolica SIN freno dice que nunca mueren:", sin_freno, "de", len(r))
print("con declinacion terminal:                                 ", con_freno, "de", len(r))

**Lectura**: ninguna de las dos curvas es «la buena». La exponencial se queda corta, la
hiperbólica se pasa de largo, y el trabajo del ingeniero es saber **dónde ponerle el
freno** — porque la fórmula sola no lo sabe.

## 5.1 · El sesgo del 7 %, en el idioma de gerencia

Un 7 % suena a nada. Veamos cuánto es en dinero.

In [ ]:
# cada barril que no se declara vale el MARGEN, no el precio:
# el precio menos lo que cuesta sacarlo
r["musd"] = (r.exp - r.real) * MARGEN / 1e6      # millones de USD

print("valor mal declarado por usar la exponencial:")
print("  mediana por campo: USD", round(r.musd.median()), "millones")
print("  en los 54 campos:  USD", round(abs(r.musd.sum()) / 1000, 1), "mil millones")
print()
print("  en DRAUGEN:        USD", round(r[r.campo == 'DRAUGEN'].musd.iloc[0]), "millones")

plt.figure(figsize=(9, 3.4))
plt.hist(r.musd, bins=16, color="#C82B40", alpha=0.72)
plt.axvline(0, color="#2D2D2D", lw=2)
plt.axvline(r.musd.median(), color="#6B1525", lw=2.4, ls="--")
plt.xlabel("valor mal declarado por campo [millones de USD]")
plt.ylabel("cantidad de campos")
plt.title("Lo que cuesta el sesgo")
plt.show()

**Lectura**: y como el sesgo va **siempre para el mismo lado**, en una cartera de campos
**no se compensa: se suma**. Ahí está la diferencia práctica entre error y sesgo, en
dólares.

### Qué mueve una decisión y qué no

| el número | cuánto vale | qué decisión mueve |
|---|---|---|
| El sesgo de la exponencial | ~USD 264 M por campo | Cuántas reservas se declaran |
| La curva y su freno | 19 vs 54 años de vida | Cuándo se cierra el campo |
| El ajuste fino de *qi* | centavos por barril | **Ninguna** — y es en lo que más tiempo se gasta |

> **La lección práctica**: antes de afinar un parámetro, pregúntense **cuántos dólares
> mueve**. Si la respuesta es «no sé», ese no es el parámetro que hay que afinar.

---

# 6 · Cuando el número no alcanza

Si el mejor método disponible se equivoca ~10 %, **¿por qué entregamos un solo número?**

Un número solo esconde justo lo que el que decide necesita: **cuánto puede moverse**.
La industria ya tiene un lenguaje para esto — **P10, P50, P90** — y lo que casi nunca
se hace es **comprobar si esos números dicen la verdad**.

### Qué significan

| nombre | qué dice | cómo se lee en voz alta |
|---|---|---|
| **P10** | El caso bajo | «Estoy bastante seguro de que produce *al menos* esto» |
| **P50** | La mitad | «Es igual de probable que quede por arriba o por abajo» |
| **P90** | El caso alto | «Sería una sorpresa que produjera *más* que esto» |

**La promesa, dicha con precisión:** una banda de P10 a P90 promete que
**8 de cada 10 veces la realidad cae adentro**. Es una afirmación *verificable*.

## 5.1 · De dónde sacamos la banda: la experiencia ajena

Un campo nuevo no es el primer campo del mundo. Hay **53 campos** que ya recorrieron el
camino completo. En vez de preguntarle a la fórmula, preguntémosle a ellos:

1. Para cada campo terminado, miramos cuánto produjo en sus primeros 5 años y cuánto
   terminó produciendo en 12.
2. La división de esos dos números es un **multiplicador**.
3. Con 53 multiplicadores tenemos una **distribución**, no un número. De ahí salen
   el P10, el P50 y el P90.

In [ ]:
# el multiplicador de cada campo
r["k"] = r.real / r.acum5

print("multiplicador (acumulado 12 anios / acumulado 5 anios):")
print("  P10:", round(r.k.quantile(0.10), 2), "x")
print("  P50:", round(r.k.quantile(0.50), 2), "x")
print("  P90:", round(r.k.quantile(0.90), 2), "x")

plt.figure(figsize=(8, 3.2))
plt.hist(r.k, bins=14, color="#2563EB", alpha=0.72)
for qq, color in [(0.10, "#EA580C"), (0.50, "#6B1525"), (0.90, "#EA580C")]:
    plt.axvline(r.k.quantile(qq), color=color, lw=2.2,
                ls="--" if qq != 0.50 else "-")
plt.xlabel("cuantas veces el acumulado de 5 anios termino siendo el de 12")
plt.ylabel("cantidad de campos")
plt.title("Lo que dice la experiencia ajena")
plt.show()

> ⚠️ **La regla que no se negocia**: al evaluar un campo, **ese campo queda fuera** del
> cálculo de los análogos. Si se deja adentro, se está usando su propio futuro para
> predecir su futuro. Es la misma trampa del `GroupKFold` de la Clase 2, con otro disfraz.

In [ ]:
# la banda para Draugen, SACANDO a Draugen del grupo
fila = r[r.campo == "DRAUGEN"].iloc[0]
otros = r[r.campo != "DRAUGEN"]          # <- aca esta la regla

P10 = fila.acum5 * otros.k.quantile(0.10)
P50 = fila.acum5 * otros.k.quantile(0.50)
P90 = fila.acum5 * otros.k.quantile(0.90)

print("Campo DRAUGEN")
print(f"  Arps exponencial : {fila.exp/1e6:5.1f} MMbbl  ({fila.e_exp:+.0f} %)")
print(f"  Arps hiperbolica : {fila.hip/1e6:5.1f} MMbbl  ({fila.e_hip:+.0f} %)")
print(f"  Analogos P50     : {P50/1e6:5.1f} MMbbl")
print(f"  banda P10-P90    : {P10/1e6:.0f} a {P90/1e6:.0f} MMbbl")
print(f"  LO QUE PASO      : {fila.real/1e6:5.1f} MMbbl")
print()
print("  la banda", "CONTIENE" if P10 <= fila.real <= P90 else "NO contiene", "la realidad")

---

# 6 · ¿La banda es honesta? La calibración

Que la banda haya acertado en Draugen puede ser suerte.

- ❌ **La pregunta fácil:** *«¿la banda contuvo el resultado?»* Con un solo campo siempre
  se puede contestar que sí — basta hacerla más ancha.
- ✅ **La pregunta seria:** *«cuando prometo 80 %, ¿acierto el 80 % de las veces?»*
  Ni más ni menos. Una banda que acierta el 100 % es tan **inútil** como una que acierta
  el 40 %: la primera es tan ancha que no dice nada.

A eso se le llama **calibración**, y es lo único que distingue un intervalo honesto de
un adorno.

In [ ]:
# repetimos el ejercicio en los 54 campos, dejando cada uno afuera por turno
dentro = 0
for i, fila in r.iterrows():
    otros = r.drop(i)                                   # se saca a si mismo
    lo = fila.acum5 * otros.k.quantile(0.10)
    hi = fila.acum5 * otros.k.quantile(0.90)
    if lo <= fila.real <= hi:
        dentro = dentro + 1

print(f"la banda P10-P90 contuvo la realidad en {dentro} de {len(r)} campos")
print(f"o sea el {round(100 * dentro / len(r))} %, cuando prometia 80 %")

In [ ]:
# y la curva completa: para cada nivel prometido, cuanto cumplio de verdad
promete, cumple = [], []
for nivel in np.arange(0.10, 0.96, 0.05):
    a = (1 - nivel) / 2                    # las colas que quedan afuera
    ok = 0
    for i, fila in r.iterrows():
        otros = r.drop(i)
        lo = fila.acum5 * otros.k.quantile(a)
        hi = fila.acum5 * otros.k.quantile(1 - a)
        if lo <= fila.real <= hi:
            ok = ok + 1
    promete.append(100 * nivel)
    cumple.append(100 * ok / len(r))

plt.figure(figsize=(7, 4.2))
plt.plot([0, 100], [0, 100], "--", color="#9CA3AF", lw=1.6,
         label="una banda perfectamente honesta")
plt.plot(promete, cumple, "o-", color="#C82B40", lw=2.4, ms=5,
         label="nuestra banda de analogos")
plt.xlabel("lo que la banda PROMETE [%]")
plt.ylabel("lo que la banda CUMPLE [%]")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.legend(loc="lower right")
plt.title("La unica pregunta que importa de un intervalo")
plt.show()

**Lectura**: la curva roja sigue a la diagonal en todo el rango. La banda **cumple lo
que promete**.

### Cómo se reporta esto

❌ **Así no:** *«Ajustamos una declinación hiperbólica de Arps con b = 0,91 y obtuvimos
un EUR a 12 años de 13,0 MMbbl.»* Suena preciso y no lo es.

✅ **Así sí:** *«Lo más probable son 12 millones de barriles en 12 años, y con 8 de cada
10 de confianza está entre 10 y 17. Ese rango lo verificamos contra 54 campos que ya
terminaron: la banda acertó el 76 % de las veces cuando prometía 80 %. Si el proyecto no
cierra con 10, no cierra.»*

### Lo que esto NO autoriza a decir

- ❌ **No sirve para un campo sin análogos.** La calibración se hereda del grupo.
- ❌ **No incluye decisiones futuras.** Una campaña de perforación cambia todo.
- ❌ **54 campos son pocos.** El P10 y el P90 se estiman en las colas, que es donde
  menos datos hay.
- ✅ **Sí sirve** para poner un rango defendible y auditado alrededor de un número que
  hoy se firma solo.

---

# 🧩 Práctica: firmen un número  ⏱️ *25 minutos*

Les toca el campo **GULLFAKS**: 32 años de historia después del pico y un pico de
569 mil barriles por día. Es de los grandes.

**El primer paso ya está resuelto.**

In [ ]:
# ── PASO 1 (RESUELTO) ── la serie de Gullfaks, y lo que se ve al firmar
MI_CAMPO = "GULLFAKS"

q_mio = campos[MI_CAMPO]
visible = q_mio.iloc[:AJUSTE]              # los 5 anios que se ven

print(MI_CAMPO, "-", len(q_mio), "meses de historia")
print("pico:", round(q_mio.max()), "bbl/dia")

plt.figure(figsize=(10, 3.4))
plt.plot(np.arange(AJUSTE) / 12, visible / 1000, color="#C82B40", lw=2)
plt.axvspan(AJUSTE / 12, 12, color="#E5E7EB")
plt.xlim(0, 12)
plt.xlabel("anios desde el pico")
plt.ylabel("miles de bbl/dia")
plt.title(f"{MI_CAMPO}: lo que se ve al momento de firmar")
plt.show()

### Paso 2 — la exponencial

Ajusten `ajustar_exponencial(q_mio)`. ¿Cuál es su D anual?
¿Declina rápido o lento comparado con la mediana de los 54 campos?

In [ ]:
# Escribe tu solucion aqui

### Paso 3 — la hiperbólica

Ajusten `ajustar_hiperbolica(q_mio)`. ¿Qué `b` les dio? ¿Este campo tiene cola?

In [ ]:
# Escribe tu solucion aqui

### Paso 4 — la banda de análogos

Construyan el P10, P50 y P90 — **acordándose de sacar a Gullfaks del grupo**.

In [ ]:
# Escribe tu solucion aqui

### Paso 5 — destapen y comparen

Usen `acumulado_real(MI_CAMPO, OBJETIVO)`.
¿Cuál de los tres estuvo más cerca? ¿La banda contuvo la realidad?

In [ ]:
# Escribe tu solucion aqui

### Paso 6 — la fecha de cierre y el dinero

Con `mes_de_abandono` y `hiperbolica_con_freno`: ¿en qué año cierra Gullfaks según cada
curva? ¿Cuántos millones de dólares hay de diferencia entre las dos respuestas?

In [ ]:
# Escribe tu solucion aqui

### Paso 7 — la pregunta de negocio

Escriban **dos frases** para el comité de reservas:

1. La primera dice **el número que firman** y su rango.
2. La segunda dice **qué haría falta para que ese rango se angoste** — porque eso es lo
   que les van a preguntar.

*(escriban sus dos frases acá)*

**Nuestra respuesta:**

---

# Cierre

| Lo que aprendimos | Dónde se usa mañana |
|---|---|
| **Arps** (1945): exponencial, hiperbólica, armónica | Toda declaración de reservas |
| **D** es la tasa de declinación; **b** es la cola | Cualquier evaluación de un activo |
| **Error** y **sesgo** no son lo mismo | Todo modelo, siempre |
| **P10/P50/P90** es una promesa | Comités de reservas, evaluaciones económicas |
| **Calibración**: la única prueba de que un intervalo es honesto | Todo pronóstico con banda |

**El número del día: 76 %.** Lo que cumplió una banda que prometía 80 %.

> **Si se llevan una sola cosa:** un pronóstico sin banda no es un pronóstico, es una
> opinión con decimales. Y una banda sin calibrar tampoco vale: cualquiera puede
> inventar un rango lo bastante ancho como para no equivocarse nunca.

---

### La próxima clase

**Módulo 5 · Clase 4 — El agua y el gas.** La segunda causa de la declinación, de frente:
corte de agua, WOR y GOR, y cuándo llega el agua a un pozo. El agua es el costo número
uno de un campo maduro.

---

### Datos

**Sokkeldirektoratet** (Norwegian Offshore Directorate) — *Production figures, monthly by
field*. Datos abiertos, `factpages.sodir.no`. Es la misma fuente del campo Volve del
Módulo 1.

**Arps, J. J. (1945).** *Analysis of Decline Curves*. Transactions of the AIME,
160(01), 228–247.